# PhoBERT NER — Colab cho team (train trên nhánh đã tạo ở local)

**Trước khi mở Colab**, làm ở máy local của bạn (không cần GPU, chỉ cần git):
```bash
git clone https://github.com/letuanlocc/training_model.git
cd training_model
git checkout -b ten/data          # đổi "ten" thành tên bạn
# Tạo/sửa file phobert_ner/data_build/handwritten_<ten>.txt, viết câu theo format [text|TAG]
git add phobert_ner/data_build/handwritten_<ten>.txt
git commit -m "Thêm câu dữ liệu"
git push -u origin ten/data
```

Sau khi đã push nhánh lên GitHub, mở Colab và chạy các cell dưới — chỉ cần sửa `BRANCH` ở Bước 1
đúng tên nhánh bạn vừa push, Colab sẽ tự lấy đúng file dữ liệu bạn đã tạo.

`Runtime → Change runtime type → T4 GPU` trước khi chạy.

In [ ]:
#@title 1. Clone đúng nhánh của bạn + cd vào project
BRANCH = "ten/data"  #@param {type:"string"}

!git clone -b {BRANCH} https://github.com/letuanlocc/training_model.git /content/training_model
%cd /content/training_model/phobert_ner
!ls data_build/handwritten*.txt

In [ ]:
#Nếu đã clone rồi thì chỉ cần pull
%cd /content/training_model
!git pull

Lệnh `!ls data_build/handwritten*.txt` phải thấy file `handwritten_<ten>.txt` của bạn trong danh sách.
Nếu không thấy → nhánh `BRANCH` chưa đúng tên, hoặc bạn chưa push file đó từ local (xem lại bước chuẩn bị ở trên).

In [ ]:
#@title 2. Cài môi trường (Java + Python packages + VnCoreNLP)
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null

# Đảm bảo cài đặt các thư viện cần thiết bằng cách trỏ đúng file requirements_working.txt
!grep -v '^#' /content/training_model/phobert_ner/requirements_working.txt | grep -v -E '^(numpy|pandas|matplotlib|scikit-learn)' | sed '/^$/d' > /tmp/req_colab.txt
!pip -q install -r /tmp/req_colab.txt
!pip -q install py_vncorenlp pytorch-crf

import os
VN = os.path.abspath('/content/training_model/vncorenlp'); B = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master'
os.makedirs(f'{VN}/models/wordsegmenter', exist_ok=True)
for f in ['VnCoreNLP-1.2.jar', 'VnCoreNLP-1.1.1.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
    !wget -q -nc -O {VN}/{f} {B}/{f}
print('Xong.')

## 3. Sinh dữ liệu + Train

Chọn 1 kiến trúc (softmax nhanh nhất, crf/lstm_crf thường chính xác hơn). ~15-30 phút trên GPU T4.

In [ ]:
ARCH = "softmax"  #@param ["softmax", "crf", "lstm_crf"]

!python data_build/build_data.py
!python data_build/validate_data.py
!bash train_{ARCH}.sh 2>&1 | grep -v 'it/s]'

## 4. Lưu model lên Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_NAME = f"{ARCH}_{BRANCH.replace('/', '_')}"  #@param {type:"raw"}

!mkdir -p "/content/drive/MyDrive/phobert_ner_outputs"
!cp -r outputs/{ARCH} "/content/drive/MyDrive/phobert_ner_outputs/{SAVE_NAME}"
print(f"Đã lưu vào /content/drive/MyDrive/phobert_ner_outputs/{SAVE_NAME}")

## 5. (Tuỳ chọn) Thử model với câu của bạn

In [ ]:
CAU_CAN_THU = "tôi muốn làm gia sư ở gần quận hải châu, tôi rảnh tối"  #@param {type:"string"}

import subprocess
cmd = f'printf "%s\\n" "{CAU_CAN_THU}" | python VPhoBertTagger/main.py predict --model_path outputs/{ARCH}/best_model.pt'
out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
for line in out.stdout.splitlines():
    if 'EOFError' not in line:
        print(line)

---
### Lỗi hay gặp
- `fatal: Remote branch <BRANCH> not found` → bạn chưa push nhánh đó lên GitHub từ local, hoặc gõ sai tên nhánh.
- `ls: cannot access 'data_build/handwritten*.txt'` sau Bước 1, không thấy file của bạn → cùng nguyên nhân trên.
- `ModuleNotFoundError` bất kỳ module nào / `vncorenlp jar not found` / Colab tự ngắt kết nối giữa buổi
  → chạy lại cell 1 và 2 (runtime bị reset, mất hết file + package đã cài).